In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
# 1 — Installation des dépendances
!pip install -q \
    transformers \
    sentence-transformers \
    faiss-cpu \
    pymupdf \
    beautifulsoup4 \
    accelerate


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 113.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.1/24.1 MB 31.6 MB/s eta 0:00:00


In [ ]:
# Avec Reranker
from sentence_transformers import CrossEncoder

reranker = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2"
)


In [4]:
# 2 — Imports & paramètres globaux
import os
import fitz
import faiss
import torch
import numpy as np
from bs4 import BeautifulSoup
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline


In [ ]:
# =========================
# PARAMÈTRES
# =========================
DATA_DIR = "/content/drive/MyDrive/PIP_2025-2026_Groupe-1_Concours/Wahib B/cleaned_json_full"





MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.2"

CHUNK_SIZE = 250

OVERLAP = 30

TOP_K = 6

SIMILARITY_THRESHOLD = 0.25

MAX_CONTEXT_CHARS = 2000
MAX_NEW_TOKENS = 150


In [6]:
import os, json
from pathlib import Path
import re

# =========================
# OUTILS
# =========================
def safe_get_text(obj):
    """
    Extrait du texte depuis différents formats JSON.
    - Si JSON concours structuré (keys concours_num + postes), on le "linéarise" en texte.
    - Sinon, fallback générique (dict/list/str).
    """
    if obj is None:
        return ""

    # --- CAS SPECIAL: JSON concours structuré ---
    if isinstance(obj, dict) and ("concours_num" in obj) and ("postes" in obj):
        lines = []

        # Métadonnées concours
        for k, label in [
            ("concours_label", "Concours"),
            ("concours_num", "Numéro"),
            ("bap", "BAP"),
            ("grade", "Grade"),
            ("emploi_type", "Emploi-type"),
            ("nb_postes", "Nombre de postes"),
            ("nb_postes_detectes", "Nombre de postes détectés"),
            ("source_file", "Source"),
        ]:
            v = obj.get(k, None)
            if v is not None and str(v).strip() != "":
                lines.append(f"{label} : {v}")

        # Détails postes
        postes = obj.get("postes", [])
        if isinstance(postes, list) and postes:
            for i, p in enumerate(postes, start=1):
                if not isinstance(p, dict):
                    continue
                lines.append(f"\nPOSTE {i} :")
                # On prend toutes les infos du poste, même si les clés varient
                for pk, pv in p.items():
                    if pv is None:
                        continue
                    # listes -> concat
                    if isinstance(pv, list):
                        pv = " ; ".join(str(x) for x in pv if str(x).strip())
                    # dict -> string simple
                    elif isinstance(pv, dict):
                        pv = " ; ".join(f"{a}={b}" for a, b in pv.items() if str(b).strip())
                    else:
                        pv = str(pv)

                    pv = pv.strip()
                    if pv:
                        lines.append(f"- {pk} : {pv}")

        return "\n".join(lines).strip()

    # --- FALLBACK GENERIQUE ---
    if isinstance(obj, str):
        return obj.strip()

    if isinstance(obj, list):
        parts = []
        for it in obj:
            t = safe_get_text(it)
            if t:
                parts.append(t)
        return "\n".join(parts).strip()

    if isinstance(obj, dict):
        # si jamais un JSON "texte" existe
        for k in ["text", "content", "clean_text", "raw_text", "body", "page_content"]:
            if k in obj and isinstance(obj[k], str) and obj[k].strip():
                return obj[k].strip()

        parts = []
        for v in obj.values():
            t = safe_get_text(v)
            if t:
                parts.append(t)
        return "\n".join(parts).strip()

    return ""


def chunk_text(text, chunk_size=250, overlap=30):
    """
    Chunk par mots, avec overlap.
    """
    words = text.split()
    step = max(1, chunk_size - overlap)
    for i in range(0, len(words), step):
        chunk = " ".join(words[i:i + chunk_size]).strip()
        if chunk:
            yield chunk



def chunk_concours_text(text, chunk_size=250, overlap=30):
    # split par postes
    parts = re.split(r"\nPOSTE\s+\d+\s*:", text)
    if len(parts) <= 1:
        # fallback chunk mots
        yield from chunk_text(text, chunk_size, overlap)
        return

    header = parts[0].strip()
    postes = parts[1:]

    # chunk header
    if header:
        yield from chunk_text(header, chunk_size, overlap)

    # chunk chaque poste
    for i, p in enumerate(postes, start=1):
        p = f"POSTE {i} :\n{p.strip()}"
        yield from chunk_text(p, chunk_size, overlap)



# =========================
# 1) CHECK DOSSIER
# =========================
print("DATA_DIR exists:", os.path.isdir(DATA_DIR))

print("Exemples fichiers:", os.listdir(DATA_DIR)[:10])

# =========================
# 2) CHARGEMENT JSON
# =========================
json_files = sorted([p for p in Path(DATA_DIR).glob("*.json")])
print("Nb fichiers JSON:", len(json_files))

documents = []
skipped = 0

for fp in json_files:
    try:
        with open(fp, "r", encoding="utf-8") as f:
            obj = json.load(f)

        full_text = safe_get_text(obj)
        if not full_text:
            skipped += 1
            continue

        # Métadonnées source
        source = fp.name

        # Déduire la page depuis le nom du fichier (ex: page_009.json -> 9)
        page = "N/A"
        if fp.stem.startswith("page_"):
            try:
                page = int(fp.stem.split("_")[1])
            except:
                page = "N/A"

        # Chunk
        for chunk in chunk_concours_text(full_text, CHUNK_SIZE, OVERLAP):
          documents.append({

              "text": chunk,
              "source": source,
              "page": page
    })


    except Exception as e:
        skipped += 1
        print(f"⚠️ Erreur fichier {fp.name}: {e}")


print(f"✅ Chunks créés: {len(documents)}")
print(f"⚠️ Fichiers ignorés/erreurs: {skipped}")

# =========================
# 3) APERÇU
# =========================
if documents:
    print("\n--- Exemple chunk ---")
    print("SOURCE:", documents[0]["source"])
    print(documents[0]["text"][:800])


DATA_DIR exists: True
Exemples fichiers: ['page_009.json', 'page_026.json', 'page_048.json', 'page_102.json', 'page_049.json', 'page_072.json', 'page_065.json', 'page_056.json', 'page_080.json', 'page_051.json']
Nb fichiers JSON: 127
✅ Chunks créés: 735
⚠️ Fichiers ignorés/erreurs: 0

--- Exemple chunk ---
SOURCE: guide_candidat_2025.json
CNRS – Guide candidat(e) 2025 (IT) Guide candidat 2025.pdf CONCOURS EXTERNES DES PERSONNELS INGÉNIEURS ET TECHNICIENS Le guide du candidat et de la candidate Edition 2025 Direction de la publication : Antoine Petit Direction de la rédaction : Hélène Maury Direction adjointe de la rédaction : Christiane Ename – Laetitia Navarro -Service recrutement et intégration (SeRI) Autrices : Dominique Marx - Emilie Faure - Nathalie Nioucel Mai 2025 5 - 6 Pourquoi candidater ? 7 - 8 Le choix des concours 9 - 10 L’inscription 11 Comment concourir ? 12 Les conditions pour concourir 13 - 14 Le déroulement des concours 15-16 Les épreuves 17 La publication des résultat

In [7]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [8]:
# 3 — Détection GPU automatique
def can_use_gpu(min_free_gb=4):
    if not torch.cuda.is_available():
        return False
    free, total = torch.cuda.mem_get_info()
    return free / (1024**3) >= min_free_gb

USE_GPU = can_use_gpu()
print(f"🔍 Mode sélectionné : {'GPU' if USE_GPU else 'CPU'}")


🔍 Mode sélectionné : CPU


In [9]:
import json
from pathlib import Path

def safe_get_text(obj):
    if isinstance(obj, dict):
        for k in ["text", "content", "clean_text", "body"]:
            if k in obj and isinstance(obj[k], str):
                return obj[k].strip()
        for v in obj.values():
            t = safe_get_text(v)
            if t:
                return t
    if isinstance(obj, list):
        return "\n".join(filter(None, (safe_get_text(x) for x in obj)))
    if isinstance(obj, str):
        return obj.strip()
    return ""


def chunk_text(text):
    words = text.split()
    step = max(1, CHUNK_SIZE - OVERLAP)
    for i in range(0, len(words), step):
        yield " ".join(words[i:i + CHUNK_SIZE])


documents = []

json_files = list(Path(DATA_DIR).glob("*.json"))
print("📁 Fichiers JSON trouvés :", len(json_files))

for fp in json_files:
    with open(fp, "r", encoding="utf-8") as f:
        obj = json.load(f)

    full_text = safe_get_text(obj)
    if not full_text.strip():
        continue

    for chunk in chunk_text(full_text):
        documents.append({
            "text": chunk,
            "source": fp.name
        })

print(f"📄 Documents indexés : {len(documents)} chunks")


📁 Fichiers JSON trouvés : 127
📄 Documents indexés : 127 chunks


In [10]:
# 5 — Embeddings & FAISS
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

assert len(documents) > 0, "documents est vide — vérifie le chargement/chunking avant FAISS."

embedder = SentenceTransformer(
    "intfloat/multilingual-e5-base",
    device="cuda" if USE_GPU else "cpu"
)

texts = ["passage: " + d["text"] for d in documents]

embeddings = embedder.encode(
    texts,
    normalize_embeddings=True,   # cosine similarity avec IndexFlatIP
    batch_size=16,
    show_progress_bar=True
)


embeddings = np.asarray(embeddings, dtype="float32")
embeddings = np.ascontiguousarray(embeddings)  # FAISS aime le contigu

dim = embeddings.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(embeddings)

print(f"✅ FAISS prêt — {index.ntotal} vecteurs, dim={dim}")


Batches:   0%|          | 0/8 [00:00<?, ?it/s]

✅ FAISS prêt — 127 vecteurs, dim=768


In [11]:
import numpy as np

def retrieve(question):
    # 1) retrieval large (E5 -> "query:")
    q_emb = embedder.encode(["query: " + question], normalize_embeddings=True)
    q_emb = np.asarray(q_emb, dtype="float32")
    q_emb = np.ascontiguousarray(q_emb)

    scores, indices = index.search(q_emb, TOP_K * 5)

    candidates = []
    for score, idx in zip(scores[0], indices[0]):
        if idx == -1:
            continue
        candidates.append({**documents[idx], "retrieval_score": float(score)})

    if not candidates:
        return []

    # 2) rerank (PAS de préfixes pour le reranker)
    pairs = [(question, c["text"]) for c in candidates]
    rerank_scores = reranker.predict(pairs)

    reranked = sorted(zip(rerank_scores, candidates), key=lambda x: x[0], reverse=True)

    # 3) garde TOP_K
    results = []
    for s, c in reranked[:TOP_K]:
        c2 = dict(c)
        c2["rerank_score"] = float(s)
        results.append(c2)

    return results

import re

def lexical_fallback(question, documents, k=4):
    q = question.lower()
    tokens = re.findall(
        r"(concours\s*n[°o]\s*\d+|poste\s*n[°o]\s*\d+|bap\s*[a-z])",
        q
    )
    if not tokens:
        return []

    hits = []
    for d in documents:
        t = d["text"].lower()
        if any(tok in t for tok in tokens):
            hits.append(d)

    return hits[:k]


In [1]:
import os
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline


USE_GPU = torch.cuda.is_available()

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

if USE_GPU:
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        device_map="auto",
        torch_dtype=torch.float16,
        low_cpu_mem_usage=True
    )

    pipe = pipeline(
        "text-generation",
        model=model,
        tokenizer=tokenizer,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=False,
        return_full_text=False,
        pad_token_id=tokenizer.eos_token_id
    )
else:
    os.environ["CUDA_VISIBLE_DEVICES"] = ""
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        device_map={"": "cpu"},
        torch_dtype=torch.float32,
        low_cpu_mem_usage=True
    )
    torch.set_num_threads(4)

    pipe = pipeline(
        "text-generation",
        model=model,
        tokenizer=tokenizer,
        device=-1,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=False,
        return_full_text=False,
        pad_token_id=tokenizer.eos_token_id
    )

print("✅ Modèle + pipeline prêts :", MODEL_NAME, "| GPU" if USE_GPU else "| CPU")


NameError: name 'MODEL_NAME' is not defined

In [ ]:
# 8 — Prompt CNRS STRICT (anti-hallucination)
def build_prompt(question, contexts):
    def fmt_source(c):
        page = c.get("page", None)
        if page is None or page == "N/A":
            return f"{c.get('source', 'inconnu')}"
        return f"{c.get('source', 'inconnu')} | page {page}"

    # Contexte (on limite chaque extrait + on limite le total)
    blocks = []
    total_chars = 0

    for c in contexts:
        excerpt = (c.get("text", "") or "").strip()
        if not excerpt:
            continue

        excerpt = excerpt[:MAX_CONTEXT_CHARS]
        block = f"- {fmt_source(c)}\n  {excerpt}"

        if total_chars + len(block) > MAX_CONTEXT_CHARS * max(1, TOP_K):
            break

        blocks.append(block)
        total_chars += len(block)

    sources_block = "\n".join(blocks) if blocks else "- (aucun contexte)"

    return f"""<s>[INST] Tu es un agent officiel d'information sur les concours ingénieur du CNRS.

RÈGLES ABSOLUES :
- Tu utilises EXCLUSIVEMENT les sources ci-dessous.
- Tu ne déduis rien.
- Tu ne complètes rien.
- Tu ne poses pas de nouvelle question.
- Tu ne réponds qu'UNE SEULE FOIS.
- Tu réponds uniquement en français.

FORMAT DE SORTIE OBLIGATOIRE :

RÉPONSE :
<réponse factuelle>

SOURCES :
- <fichier> | page <numéro>

SI l'information n'est PAS clairement présente, répond EXACTEMENT :

RÉPONSE :
Je ne dispose pas de cette information dans les documents de référence.

SOURCES :
Aucune

SOURCES DISPONIBLES :
{sources_block}

QUESTION :
{question} [/INST]
"""


In [ ]:
# Bloquer / nettoyer la sortie du modèle
def clean_output(text):
    stop_markers = [
        "\nQUESTION :",
        "\n❓",
        "\n[INST]",
        "\nSOURCES DISPONIBLES",
        "\nSOURCES :",
        "\nSOURCE :",
    ]

    for marker in stop_markers:
        if marker in text:
            text = text.split(marker)[0]

    return text.strip()


In [ ]:
# 9 — Fonction answer() finale

REFUS = "Je ne dispose pas de cette information dans les documents de référence."

def answer(question):
    contexts = retrieve(question)

    # 🔁 fallback lexical si retrieval vide
    if not contexts:
        contexts = lexical_fallback(question, documents, k=TOP_K)

    # Toujours rien => refus
    if not contexts:
        return f"RÉPONSE :\n{REFUS}\n\nSOURCES :\nAucune"

    prompt = build_prompt(question, contexts)

    # Génération
    gen = pipe(prompt)[0]["generated_text"]
    output = clean_output(gen)

    # Si le modèle refuse ou ne respecte pas le format, on force un refus propre
    if (REFUS in output) or ("RÉPONSE :" not in output):
        return f"RÉPONSE :\n{REFUS}\n\nSOURCES :\nAucune"

    return output


In [ ]:
# 10 — Mode interactif (démo)
print("\n🤖 Agent CNRS prêt. Tape 'quitter' pour quitter.\n")

while True:
    q = input("❓ Question : ").strip()

    if not q:
        continue

    if q.lower() in {"quitter", "quit", "exit"}:
        break

    print("\n" + answer(q))
    print("\n" + "-" * 60)


# Sauvegarde du modele

In [ ]:
# 1️ Créer un dossier de sauvegarde
SAVE_DIR = "/content/drive/MyDrive/PIP_2025-2026_Groupe-1_Concours/Wahib B"
os.makedirs(SAVE_DIR, exist_ok=True)



In [ ]:
# 2️ Sauvegarder le modèle LLM + tokenizer
model.save_pretrained(f"{SAVE_DIR}/llm")
tokenizer.save_pretrained(f"{SAVE_DIR}/llm")

In [ ]:
# 3️ Sauvegarder l’embedder
embedder.save(f"{SAVE_DIR}/embedder")


In [ ]:
# 4 - Sauvegarder les documents
import json

with open(f"{SAVE_DIR}/documents.json", "w", encoding="utf-8") as f:
    json.dump(documents, f, ensure_ascii=False, indent=2)


In [ ]:
# 5️ Sauvegarder l’index FAISS
faiss.write_index(index, f"{SAVE_DIR}/faiss.index")


# Metrique

# Atelnative sans ragas

In [ ]:
from sentence_transformers import util
import numpy as np

def context_precision(question, context_texts, embedder, threshold=0.3):
    if not context_texts:
        return 0.0

    q_emb = embedder.encode([question], normalize_embeddings=True)
    c_embs = embedder.encode(context_texts, normalize_embeddings=True)

    sims = util.cos_sim(q_emb, c_embs)[0].numpy()
    return float(np.mean(sims >= threshold))


def context_recall(ground_truth, context_texts, embedder, threshold=0.3):
    if not context_texts or not ground_truth:
        return 0.0

    gt_emb = embedder.encode([ground_truth], normalize_embeddings=True)
    c_embs = embedder.encode(context_texts, normalize_embeddings=True)

    sims = util.cos_sim(gt_emb, c_embs)[0].numpy()
    return float(np.max(sims) >= threshold)



In [ ]:
from sentence_transformers import util

def answer_relevancy(question, answer, embedder):
    """
    Mesure la pertinence sémantique de la réponse par rapport à la question
    (cosine similarity des embeddings).
    """
    if not question or not answer:
        return 0.0

    q_emb = embedder.encode([question], normalize_embeddings=True)
    a_emb = embedder.encode([answer], normalize_embeddings=True)

    score = util.cos_sim(q_emb, a_emb)[0][0].item()
    return float(score)



In [ ]:
# Fonction utilitaire
def get_context_texts(context_docs):
    """
    Transforme les documents récupérés par retrieve()
    en liste de textes (str)
    """
    if not context_docs:
        return []

    return [c.get("text", "") for c in context_docs if c.get("text")]



In [ ]:
# Jeu de questions d’évaluation (≈20 questions CNRS)
questions_test = [
    "Quel est l’emploi-type du Concours n°1 ?",
    "Combien de postes sont ouverts pour le Concours n°1 ?",
    "Où se situe l’affectation du poste n°2 du Concours n°1 ?",
    "Quelle est la mission principale du poste n°3 du Concours n°1 ?",
    "Quels concours correspondent à un profil en bioinformatique ?",
    "Quels concours relèvent de la BAP A ?",
    "Quels sont les grades des ingénieurs au CNRS ?",
    "Quels avantages sont proposés aux ingénieurs du CNRS ?",
    "Le CNRS propose-t-il du télétravail ?",
    "Existe-t-il un accompagnement social au CNRS ?",
    "Quelles sont les phases d’un concours ingénieur CNRS ?",
    "Quelles sont les conditions d’accès aux concours ingénieur CNRS ?",
    "Comment se déroule une audition de concours ?",
    "Les concours CNRS sont-ils nationaux ou locaux ?",
    "Quel est le niveau d’anglais exigé pour le Concours n°2 ?",
    "Quel est le groupe de fonctions du poste n°1 du Concours n°1 ?",
    "Quel est le contexte scientifique du poste situé à Lyon ?",
    "Quel concours conviendrait à un profil en analyse de données ?",
    "Quel est le salaire net exact d’un ingénieur CNRS débutant ?",
    "Quel temps fait-il aujourd’hui au CNRS de Lyon ?"
]

results = []  # structure propre pour l'évaluation

answers = []
placeholder_contexts = []

for q in questions_test:
    contexts = retrieve(q)
    context_texts = get_context_texts(contexts)

    answer_text = answer(q)

    answers.append(answer_text)
    placeholder_contexts.append(context_texts)


    results.append({
        "question": q,
        "answer": answer_text,
        "contexts": context_texts,
        "nb_contexts": len(context_texts)
    })




In [ ]:
# Ground Truth
ground_truths = [
    "L’emploi-type du Concours n°1 est Ingenieure ou ingenieur biologiste en analyse de données.",
    "Le Concours n°1 propose trois postes.",
    "Le poste n°2 du Concours n°1 est situé à Paris 15.",
    "La mission principale du poste n°3 concerne l’analyse de données en bioinformatique.",
    "Des concours en bioinformatique sont proposés dans la BAP A.",
    "La BAP A correspond aux sciences du vivant.",
    "Les grades incluent ingénieur d’études et ingénieur de recherche.",
    "Les avantages incluent télétravail, accompagnement social et mobilité.",
    "Le télétravail est possible au CNRS.",
    "Un accompagnement social est proposé aux agents du CNRS.",
    "Les phases comprennent sélection sur dossier et audition.",
    "Les conditions d’accès dépendent du concours et du grade.",
    "Une audition est organisée devant un jury.",
    "Les concours sont organisés au niveau national.",
    "Le niveau d’anglais n’est pas explicitement indiqué.",
    "Le poste appartient au groupe de fonctions 3.",
    "Le contexte scientifique concerne la recherche en infectiologie.",
    "Des concours existent pour l’analyse de données.",
    "Le système ne dispose pas de cette information.",
    "Le système ne dispose pas de cette information."
]



In [ ]:
import pandas as pd

scores = []

for q, a, ctx_texts, gt in zip(
    questions_test,
    answers,
    placeholder_contexts,
    ground_truths
):
    scores.append({
        "context_precision": context_precision(q, ctx_texts, embedder),
        "context_recall": context_recall(gt, ctx_texts, embedder),
        "answer_relevancy": answer_relevancy(q, a, embedder),
        "nb_contexts": len(ctx_texts)
    })

df = pd.DataFrame(scores)
display(df)

print("\n📊 Scores moyens :")
print(df.mean(numeric_only=True))
